In [ ]:
import myFunctions as myF
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Set font configurations for Matplotlib (Times New Roman for English, SimHei for Chinese)
plt.rcParams['font.family'] = ['Times New Roman', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# --- Load dataset ---

In [ ]:
# Note: Loading only the first 88,000 rows as per original script
df = pd.read_csv('./merged_combined_data.csv').iloc[:88000]

# Feature Engineering: Extract Month and Day
df['转化月份和日'] = pd.to_datetime(df['日期'], format='ISO8601')
df['月份'] = df['转化月份和日'].dt.month
df['日'] = df['转化月份和日'].dt.day
df.drop(columns=['转化月份和日'], inplace=True)

# --- Descriptive Statistics ---

In [ ]:
# Define the feature set for analysis
columns = ['序号', '非市场化机组出力(MW)', '水电实际值(MW)', '风电实际值(MW)', '光伏实际值(MW)',
       '新能源总加实际值(MW)', '实时出清价格(元/MWh)', '日前出清价格(元/MWh)', '检修容量(MW)',
       '日前联络线电力值(MW)', '省内负荷预测电力值(MW)', '日前联络线和省内负荷预测总加电力值(MW)',
       '统调用电负荷电力值(MW)', '新能源出力预测电力值(MW)', '新能源出力预测风电(MW)', '新能源出力预测光伏(MW)',
       't2m', 'u10', 'v10', 'tp', '月份', '日']
print(f"Number of columns: {len(columns)}")

# Calculate Mean and Standard Deviation for the dataset
stats = pd.DataFrame({
    'Mean': df[columns].mean(),
    'Standard Deviation': df[columns].std()
})

# Display statistics (in a notebook environment)
print(stats) 

In [ ]:
# bins = 8
columns = ['实时出清价格(元/MWh)', '日前出清价格(元/MWh)','新能源出力预测电力值(MW)']

X = df[columns].copy()
for i in range(1, 17):
    X[f'{i}'] = df['实时出清价格(元/MWh)'].shift(i)
columns = X.columns
X = X.fillna(X.mean())
# calculate SU matrix
su_matrix = pd.DataFrame(index=columns, columns=columns)

for col1 in columns:
    for col2 in columns:
        su_matrix.loc[col1, col2] = myF.SymmetricUncertainty().cal_symmetric_uncertainty_discrete(x=X[col1], y=X[col2])

rename_dict = {
    '实时出清价格(元/MWh)': 'price',
    '日前出清价格(元/MWh)': 'a',
    '新能源出力预测电力值(MW)': 'b'
}

renamed_columns = [
    rename_dict.get(col, col) if idx < 3 else col
    for idx, col in enumerate(columns)
]


su_matrix.index = renamed_columns
su_matrix.columns = renamed_columns

# pearson martrix
pearson_matrix = X.corr(method='pearson')
# spearman_matrix = X.corr(method='spearman')

combined_matrix = np.zeros_like(su_matrix, dtype=float)
tril_indices = np.tril_indices_from(combined_matrix)
combined_matrix[tril_indices] = su_matrix.values[tril_indices]
triu_indices = np.triu_indices_from(combined_matrix, k=1)
combined_matrix[triu_indices] = pearson_matrix.values[triu_indices]

combined_matrix = pd.DataFrame(combined_matrix, index=su_matrix.index, columns=su_matrix.columns)

plt.figure(figsize=(18, 15), dpi=400)
# sns.heatmap(su_matrix.astype(float), annot=True, fmt='.2f', cmap='Greys', vmin=0, vmax=1, annot_kws={"size": 15})
# sns.heatmap(su_matrix.astype(float), annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, annot_kws={"size": 15})
sns.heatmap(combined_matrix.astype(float), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, annot_kws={"size": 15})
plt.tick_params(axis='both', which='major', labelsize=24)
# plt.title('Thermodynamic Diagram of Correlation Matrix')
plt.show()

# --- Cluster Analysis and Statistics ---

In [ ]:
# Prepare data for clustering (Drop NaNs)
data = df[columns].copy().dropna()

# Perform K-Means clustering (k=2)
# random_state is set for reproducibility
kmeans = KMeans(n_clusters=2, random_state=10, n_init=10) 
kmeans.fit(data)
cluster_labels = kmeans.labels_

# Save cluster labels to the dataframe
data['category'] = cluster_labels

# Calculate Mean and Standard Deviation for each cluster category
means = data.groupby('category').mean().T  # Transpose to have metrics as rows
stds = data.groupby('category').std().T    

# Merge statistics into a single DataFrame
stats = pd.concat([means, stds], axis=1)
stats.columns = ['Mean_0', 'Mean_1', 'Standard Deviation_0', 'Standard Deviation_1']

# Display cluster statistics (in a notebook environment)
print(stats)

# --- Model Performance Comparison: Global vs. Clustered ---

In [ ]:
data = df[columns].copy().dropna()

print('--- Global Model Performance ---')
# Run regression analysis on the entire dataset
print(f'--- Performance for Entire Dataset ---')
myF.RegressionAnalysis().LR_DT_SVR_BT_RF_finally_and_no_picture(
    X=data.drop(columns='实时出清价格(元/MWh)'), 
    y=data['实时出清价格(元/MWh)']
)

# Re-run clustering
kmeans = KMeans(n_clusters=2, random_state=10, n_init=10) 
kmeans.fit(data)
data['category'] = kmeans.labels_
data.dropna(inplace=True)

# Run regression analysis separately for each cluster
for i in range(2):
    print(f'--- Performance for Cluster {i+1} ---')
    # Filter data for the current cluster
    y = data[data['category'] == i]['实时出清价格(元/MWh)']
    X_use = data[data['category'] == i].drop(columns=['实时出清价格(元/MWh)', 'category'])

    myF.RegressionAnalysis().LR_DT_SVR_BT_RF_finally_and_no_picture(X=X_use, y=y)


# --- Experiment: Feature Construction (FC) ---

In [ ]:
data = df[columns].copy()
print(f"Original shape: {data.shape}")

# Define columns to create lag features for
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
             '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
             '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']

# Create lag features (shift 1 to 4 steps)
for col in shift_columns:
    for i in range(1, 5):
        # Shift the column down by 'i' steps
        data[f'{col}_{i}'] = data[f'{col}'].shift(i) 

data.dropna(inplace=True)
print(f"Shape after FC and dropna: {data.shape}")

print('--- Results using Feature Construction (FC) ---')
myF.RegressionAnalysis().LR_DT_SVR_BT_RF_finally_and_no_picture(
    X=data.drop(columns='实时出清价格(元/MWh)'), 
    y=data['实时出清价格(元/MWh)']
)

# --- Experiment: Feature Selection (FS) ---

In [ ]:
data = df[columns].copy()
print(f"Original shape: {data.shape}")

# Select features using Symmetric Uncertainty
# Note: Ensure the method name matches your module (SymmetricUncertainty vs symmetric_uncertainty)
selected_features = myF.symmetric_uncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)

print(f'Selected feature set: {selected_features}')

data.dropna(inplace=True)
print('--- Results using Feature Selection (FS) ---')
myF.RegressionAnalysis().LR_DT_SVR_BT_RF_finally_and_no_picture(
    X=data[selected_features], 
    y=data['实时出清价格(元/MWh)']
)

# --- Experiment: Hybrid Approach (FC + FS) ---

In [ ]:
data = df[columns].copy()
print(f"Original shape: {data.shape}")

# 1. Feature Construction (Lag features)
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
             '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
             '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']
for col in shift_columns:
    for i in range(1, 5):
        data[f'{col}_{i}'] = data[f'{col}'].shift(i) 

print(f"Shape after FC: {data.shape}")

# 2. Feature Selection on the expanded dataset
selected_features = myF.symmetric_uncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)

print(f'Selected feature set: {selected_features}')

data.dropna(inplace=True)
print('--- Results using Hybrid Approach (FC + FS) ---')
myF.RegressionAnalysis().LR_DT_SVR_BT_RF_finally_and_no_picture(
    X=data[selected_features], 
    y=data['实时出清价格(元/MWh)']
)